# The Monty Hall Problem
This [problem](https://en.wikipedia.org/wiki/Monty_Hall_problem) has flummoxed many people over the years, [mathematicians included](https://web.archive.org/web/20140413131827/http://www.decisionsciences.org/DecisionLine/Vol30/30_1/vazs30_1.pdf). Let's see if we can work it out by simulation.

The setting is derived from a television game show called "Let's Make a Deal". Monty Hall hosted this show in the 1960's, and it has since led to a number of spin-offs. An exciting part of the show was that while the contestants had the chance to win great prizes, they might instead end up with "zonks" that were less desirable. This is the basis for what is now known as *the Monty Hall problem*.

The setting is a game show in which the contestant is faced with three closed doors. Behind one of the doors is a fancy car, and behind each of the other two there is a goat. The contestant doesn't know where the car is, and has to attempt to find it under the following rules.

- The contestant makes an initial choice, but that door isn't opened.
- At least one of the other two doors must have a goat behind it. Monty opens one of these doors to reveal a goat, displayed in all its glory in [Wikipedia](https://en.wikipedia.org/wiki/Monty_Hall_problem):

![Three blue doors on a white background. The doors on the left and in the center are labeled with a green question mark. The third door is open revealing a goat.](../../../images/monty_hall_goat.png)

- There are two doors left, one of which was the contestant's original choice. One of the doors has the car behind it, and the other one has a goat. The contestant now gets to choose which of the two doors to open.

The contestant has a decision to make. Which door should she choose to open, if she wants the car? Should she stick with her initial choice, or switch to the other door? That is the Monty Hall problem.

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
path_data = '../../../assets/data/'


## The Solution

In any problem involving chances, the assumptions about randomness are important. It's reasonable to assume that there is a 1/3 chance that the contestant's initial choice is the door that has the car behind it. 

The solution to the problem is quite straightforward under this assumption, though the straightforward solution doesn't convince everyone. Here it is anyway.

- The chance that the car is behind the originally chosen door is 1/3.
- The car is behind either the originally chosen door or the door that remains. It can't be anywhere else.
- Therefore, the chance that the car is behind the door that remains is 2/3.
- Therefore, the contestant should switch.

That's it. End of story. 

Not convinced? Then let's simulate the game and see how the results turn out.

## Simulation ##
The simulation will be more complex that those we have done so far. Let's break it down.

**Step 1: What to Simulate**

For each play we will simulate what's behind all three doors:
- the one the contestant first picks
- the one that Monty opens
- the remaining door

So we will be keeping track of three quantitites, not just one.

**Step 2: Simulating One Play**

As is often the case in simulating a game, the bulk of the work consists of simulating one play of the game. This involves several pieces.

**The goats:** We start by setting up an array `goats` that contains unimaginative names for the two goats. 

In [2]:
goats = make_array('first goat', 'second goat')


To help Monty conduct the game, we are going to have to identify which goat is selected and which one is revealed behind the open door. The function `other_goat` takes one goat and returns the other.

In [3]:
def other_goat(x):
    if x == 'first goat':
        return 'second goat'
    elif x == 'second goat':
        return 'first goat'


In [4]:
other_goat('first goat'), other_goat('second goat'), other_goat('watermelon')


('second goat', 'first goat', None)

The string `watermelon` is not the name of one of the goats, so when `watermelon` is the input then `other_goat` does nothing.

**The options:** The array `hidden_behind_doors` contains the three things that are
behind the doors.

In [5]:
hidden_behind_doors = np.append(goats, 'car')
hidden_behind_doors


array(['first goat', 'second goat', 'car'],
      dtype='<U11')

We are now ready to simulate one play. To do this, we will define a function `monty_hall_game` that takes no arguments. When the function is called, it plays Monty's game once and returns an array consisting of:

- the contestant's guess
- what Monty reveals when he opens a door
- what remains behind the other door

The game starts with the contestant choosing one door at random. In doing so, the contestant makes a random choice from among the first goat, the second goat, and the car.

If the contestant happens to pick one of the goats, then the other goat is revealed and the car is behind the remaining door.

If the contestant happens to pick the car, then Monty reveals one of the goats and the other goat is behind the remaining door.

In [6]:
def monty_hall_game():
    """Return 
    [contestant's guess, what Monty reveals, what remains behind the other door]"""
    
    contestant_guess = np.random.choice(hidden_behind_doors)
    
    if contestant_guess == 'first goat':
        return make_array(contestant_guess, 'second goat', 'car')
    
    if contestant_guess == 'second goat':
        return make_array(contestant_guess, 'first goat', 'car')
    
    if contestant_guess == 'car':
        revealed = np.random.choice(goats)
        return make_array(contestant_guess, revealed, other_goat(revealed))


Let's play! Run the cell several times and see how the results change.

In [7]:
monty_hall_game()


array(['second goat', 'first goat', 'car'],
      dtype='<U11')

**Step 3: Number of Repetitions**

To gauge the frequency with which the different results occur, we have to play the game many times and collect the results. Let's run 10,000 repetitions.

**Step 4: Simulating Multiple Repetitions**

It's time to run the whole simulation. But unlike our previous simulations in which each simulation produced a single value, in this example we simulate an array of three values each time. 

We will treat this simulated array as a row of a DataFrame. So instead of starting with an empty array and augmenting it with each new simulated value, we will start with an empty DataFrame and
augment the DataFrame with each new simulated row. Each row will contain the complete result of one play. 

One way to grow a DataFrame by adding a new row is to use `pl.concat`. If `my_dataframe` is a DataFrame and `new_row` is an array containing the entries in a new row, then `my_dataframe = pl.concat([my_dataframe, pl.DataFrame([new_row], schema=my_dataframe.schema, orient="row")])` adds the new row to the bottom of `my_dataframe`.

`concat` returns a new DataFrame. Assigning that DataFrame back to the name `my_dataframe` is what makes the name refer to one more row than it did before.

We will start with a collection DataFrame `games` that has three empty columns. We can do this by just specifying the column labels in the same order in which `monty_hall_game` returns the result of one game.

Now we can add 10,000 rows to `games`. Each row will represent the result of one play of Monty's game.

In [8]:
# empty collection DataFrame
games = pl.DataFrame(schema={
    "Guess": pl.String,
    "Revealed": pl.String,
    "Remaining": pl.String,
})

# Play the game 10000 times and 
# record the results in the DataFrame games

for i in np.arange(10000):
    new_row = monty_hall_game()
    games = pl.concat([
        games,
        pl.DataFrame([new_row], schema=games.schema, orient="row"),
    ])


The simulation is done. Notice how short the code is. The majority of the work was done in simulating the outcome of one game.

In [9]:
games.show(3)


Guess,Revealed,Remaining
str,str,str
"""first goat""","""second goat""","""car"""
"""first goat""","""second goat""","""car"""
"""second goat""","""first goat""","""car"""


## Visualization

To see whether the contestant should stick with her original choice or switch, let's see how frequently the car is behind each of her two options.

It is no surprise that the three doors appear about equally often as the contestant's original guess.

In [10]:
original_choice = games.group_by("Guess").len(name="count").sort("Guess")
original_choice


Guess,count
str,u32
"""car""",3275
"""first goat""",3367
"""second goat""",3358


Once Monty has eliminated a goat, how often is the car behind the remaining door?

In [11]:
remaining_door = games.group_by("Remaining").len(name="count").sort("Remaining")
remaining_door


Remaining,count
str,u32
"""car""",6725
"""first goat""",1646
"""second goat""",1629


As our earlier solution said, the car is behind the remaining door two-thirds of the time, to a pretty good approximation. The contestant is twice as likely to get the car if she switches than if she sticks with her original choice.

To see this graphically, we can join the two DataFrames above and draw overlaid bar charts.

In [12]:
joined = original_choice.join(remaining_door, left_on="Guess", right_on="Remaining")
# POLARS_CHANGE: the right key is dropped and the second count column is suffixed
combined = joined.rename({
    joined.columns[0]: "Item",
    joined.columns[1]: "Original Door",
    joined.columns[2]: "Remaining Door",
})
combined


Item,Original Door,Remaining Door
str,u32,u32
"""car""",3275,6725
"""first goat""",3367,1646
"""second goat""",3358,1629


In [13]:
combined.hvplot.barh(
    "Item",
    ["Original Door", "Remaining Door"],
    color=["#002fa7", "#706700"],
)


Bar plot with labels 0 to 7000 on the x-axis and 'Item' on the y-axis. Each category has two bars, one is dark blue for 'Original Door' and the other is gold for 'Remaining Door.' Each category has the same length for the dark blue 'Original Door' bar, between 3000 and 4000. The first category is 'car' and the gold bar is twice as long as the dark blue bar, approx 7000. The next two categories are 'first goat' and 'second goat' and have the same length for the gold bars, between 1000 and 2000.

Notice how the three blue bars are almost equal – the original choice is equally likely to be any of the three available items. But the gold bar corresponding to `Car` is twice as long as the blue. 

The simulation confirms that the contestant is twice as likely to win if she switches.